In [1]:
from DDCASPT2 import DDCASPT2
import pickle, os, shutil
from glob import glob
import numpy as np
from joblib import Parallel, delayed
import pandas as pd

In [3]:
radius_range=np.linspace(0.6,3,100)
# radius_range=[0.94]
chains=np.arange(2,14,2)
# chains=np.arange(14,22,2)
print(chains)
# train_ind,test_ind=train_test_split(radius_range, test_size=0.3, random_state=0)
# print(len(train_ind),len(test_ind))
# with open('train_ind.pickle', 'wb') as handle:
#     pickle.dump(train_ind, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open('test_ind.pickle', 'wb') as handle:
#     pickle.dump(test_ind, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('test_ind.pickle', 'rb') as handle:
    test_ind = pickle.load(handle)

with open('train_ind.pickle', 'rb') as handle:
    train_ind = pickle.load(handle)
    
print(len(train_ind),len(test_ind))    


[ 2  4  6  8 10 12]
70 30


/tmp/ipykernel_21527/2467865580.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_ind = pickle.load(handle)
/tmp/ipykernel_21527/2467865580.py:18: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is th

In [4]:
# for i in chains:
topdir = os.getcwd()
basis_set='ANO-RCC-VDZP'


In [5]:
# def run(i):
#     dirname=f'H{i}_chain'
#     print(dirname)
#     if os.path.exists(dirname)==False:
#         os.mkdir(dirname)
        
#     # os.chdir(os.path.join(topdir,dirname))    
#     for idxr, r in enumerate(radius_range):
        
#         # Loop radius
#         name=f"H{i}_{r:.2f}"
        
#         # Create files
#         subdirpath = os.path.join(topdir,dirname,f'{name}')
#         if os.path.exists(subdirpath)==False:
#             os.mkdir(subdirpath)
#         if os.path.exists(os.path.join(subdirpath,f'{name}.csv'))==False:
#             shutil.rmtree(os.path.join('tmp'),name)
            
#             # Write xyz
#             with open(os.path.join(subdirpath,f'{name}.xyz'),'w') as f:
#                 f.write(f'{i}\n\n')
#                 for j in range(i):
#                     f.write(f'H {0:>8f} {0:>8f} {j*r:>8f}\n')
#             print(subdirpath)
#             if idxr==0:
#                 d = DDCASPT2(subdirpath,basis_set,name,i,i,0,previous=None)()
#             else:            
#                 previous=os.path.join(topdir,dirname,f'H{i}_{radius_range[idxr-1]:.2f}',f"H{i}_{radius_range[idxr-1]:.2f}.RasOrb")
#                 print(previous)
#                 d = DDCASPT2(subdirpath,basis_set,name,i,i,0,previous=previous)()
    
#     for idxr, r in enumerate(radius_range):
#         # Loop radius
#         name=f"H{i}_{r:.2f}" 
        
#         subdirpath = os.path.join(topdir,dirname,f'{name}')
        
        
#         for j in glob(os.path.join(subdirpath,"*GMJ*.csv"))+glob(os.path.join(subdirpath,"*Orb*"))+glob(os.path.join(subdirpath,"*h5"))+glob(os.path.join(subdirpath,"xmldump")):
#             os.remove(j)        




def run(i):
    dirname = f'H{i}_chain'
    print(dirname)
    
    if not os.path.exists(dirname):
        os.mkdir(dirname)
        
    first_valid_idx = None  # Track the first valid index
    
    for idxr, r in enumerate(radius_range):
        # Loop radius
        name = f"H{i}_{r:.2f}"
        
        # Create files
        subdirpath = os.path.join(topdir, dirname, f'{name}')
        if not os.path.exists(subdirpath):
            os.mkdir(subdirpath)
        
        if not os.path.exists(os.path.join(subdirpath, f'{name}.csv')):
            shutil.rmtree(os.path.join('tmp'), ignore_errors=True)
            
            # Write xyz
            with open(os.path.join(subdirpath, f'{name}.xyz'), 'w') as f:
                f.write(f'{i}\n\n')
                for j in range(i):
                    f.write(f'H {0:>8f} {0:>8f} {j*r:>8f}\n')
            
            print(subdirpath)

            try:
                if first_valid_idx is None:  # First attempt
                    d = DDCASPT2(subdirpath, basis_set, name, i, i, 0, casscf_previous=None)()
                    first_valid_idx = idxr  # Mark this as the first valid calculation
                else:
                    previous = os.path.join(
                        topdir, dirname, f'H{i}_{radius_range[first_valid_idx]:.2f}', 
                        f"H{i}_{radius_range[first_valid_idx]:.2f}.RasOrb"
                    )
                    print(previous)
                    d = DDCASPT2(subdirpath, basis_set, name, i, i, 0, casscf_previous=previous)()
            except Exception as e:
                print(f"Error at index {idxr} with r={r}: {e}")
                continue  # Skip to the next iteration if it fails
    
    # Cleanup section
    for idxr, r in enumerate(radius_range):
        name = f"H{i}_{r:.2f}" 
        subdirpath = os.path.join(topdir, dirname, f'{name}')
        
        for j in glob(os.path.join(subdirpath, "*GMJ*.csv")) + \
                 glob(os.path.join(subdirpath, "*Orb*")) + \
                 glob(os.path.join(subdirpath, "*h5")) + \
                 glob(os.path.join(subdirpath, "xmldump")):
            os.remove(j)


In [ ]:
Parallel(n_jobs=-1)(delayed(run)(i) for i in chains)

H2_chain
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H2_chain/H2_0.60
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'S1': 2, 'S2': 3, 'S3': 4, 'S4': 5, 'S5': 6, 'S6': 7, 'S7': 8, 'S8': 9}


Pairs: 100%|██████████| 2/2 [00:00<00:00, 226.24it/s]

Root: 100%|██████████| 1/1 [00:00<00:00, 16.42it/s]


H4_chain
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H4_chain/H4_0.60
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'S1': 4, 'S2': 5, 'S3': 6, 'S4': 7, 'S5': 8, 'S6': 9, 'S7': 10, 'S8': 11, 'S9': 12, 'S10': 13, 'S11': 14, 'S12': 15, 'S13': 16, 'S14': 17, 'S15': 18, 'S16': 19}


Pairs: 100%|██████████| 3/3 [00:00<00:00, 67.97it/s]

Root: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]


H6_chain
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H6_chain/H6_0.60
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'S1': 6, 'S2': 7, 'S3': 8, 'S4': 9, 'S5': 10, 'S6': 11, 'S7': 12, 'S8': 13, 'S9': 14, 'S10': 15, 'S11': 16, 'S12': 17, 'S13': 18, 'S14': 19, 'S15': 20, 'S16': 21, 'S17': 22, 'S18': 23, 'S19': 24, 'S20': 25, 'S21': 26, 'S22': 27, 'S23': 28, 'S24': 29}


...

Pairs: 100%|██████████| 3/3 [00:02<00:00,  1.40it/s]

Pairs:   0%|          | 0/3 [00:00<?, ?it/s]

/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H12_chain/H12_2.47
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H12_chain/H12_0.60/H12_0.60.RasOrb
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'A7': 6, 'A8': 7, 'A9': 8, 'A10': 9, 'A11': 10, 'A12': 11, 'S1': 12, 'S2': 13, 'S3': 14, 'S4': 15, 'S5': 16, 'S6': 17, 'S7': 18, 'S8': 19, 'S9': 20, 'S10': 21, 'S11': 22, 'S12': 23, 'S13': 24, 'S14': 25, 'S15': 26, 'S16': 27, 'S17': 28, 'S18': 29, 'S19': 30, 'S20': 31, 'S21': 32, 'S22': 33, 'S23': 34, 'S24': 35, 'S25': 36, 'S26': 37, 'S27': 38, 'S28': 39, 'S29': 40, 'S30': 41, 'S31': 42, 'S32': 43, 'S33': 44, 'S34': 45, 'S35': 46, 'S36': 47, 'S37': 48, 'S38': 49, 'S39': 50, 'S40': 51, 'S41': 52, 'S42': 53, 'S43': 54, 'S44': 55, 'S45': 56, 'S46': 57, 'S47': 58, 'S48': 59}



Pairs: 100%|██████████| 3/3 [00:02<00:00,  1.45it/s]

Root: 100%|██████████| 1/1 [08:26<00:00, 506.84s/it]


/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H12_chain/H12_2.49
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H12_chain/H12_0.60/H12_0.60.RasOrb
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'A7': 6, 'A8': 7, 'A9': 8, 'A10': 9, 'A11': 10, 'A12': 11, 'S1': 12, 'S2': 13, 'S3': 14, 'S4': 15, 'S5': 16, 'S6': 17, 'S7': 18, 'S8': 19, 'S9': 20, 'S10': 21, 'S11': 22, 'S12': 23, 'S13': 24, 'S14': 25, 'S15': 26, 'S16': 27, 'S17': 28, 'S18': 29, 'S19': 30, 'S20': 31, 'S21': 32, 'S22': 33, 'S23': 34, 'S24': 35, 'S25': 36, 'S26': 37, 'S27': 38, 'S28': 39, 'S29': 40, 'S30': 41, 'S31': 42, 'S32': 43, 'S33': 44, 'S34': 45, 'S35': 46, 'S36': 47, 'S37': 48, 'S38': 49, 'S39': 50, 'S40': 51, 'S41': 52, 'S42': 53, 'S43': 54, 'S44': 55, 'S45': 56, 'S46': 57, 'S47': 58, 'S48': 59}


Pairs: 100%|██████████| 3/3 [00:02<00:00,  1.38it/s]

Features:  58%|█████▊    | 84/144 [05:08<03:41,  3.69s/it]